# P77 — Contracción y selección en regresión mediante el lasso

## 1. Título y paper

**Paper:** *Regression Shrinkage and Selection via the Lasso*  
**Autoría:** Robert Tibshirani  
**Año y venue:** 1996 · Journal of the Royal Statistical Society, Series B, 58(1), 267–288  
**Nivel:** L3 · **Motor:** `lasso`  
**Ficha completa:** [`P77_lasso`](../../papers/foundational/P77_lasso/README.md)

**Hito:** Una penalización que estima y selecciona a la vez: pone coeficientes exactamente en cero.

- [doi:10.1111/j.2517-6161.1996.tb02080.x](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La regresión por mínimos cuadrados con muchas variables sobreajusta y produce modelos imposibles de interpretar. La selección por subconjuntos es inestable y la penalización de cresta encoge todos los coeficientes pero no elimina ninguno.
2. Ejecutar una implementación mínima de la propuesta: Penalizar la suma de los valores absolutos de los coeficientes. La geometría de esa restricción tiene esquinas sobre los ejes, y el óptimo tiende a caer en ellas: los coeficientes irrelevantes quedan exactamente en cero.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Hoerl y Kennard (1970), regresión de cresta
- Breiman (1995), garrote no negativo


## 4. Intuición

La regresión de cresta encoge todos los coeficientes hacia cero y no llega nunca. El lasso cambia el círculo por un rombo, y los rombos tienen esquinas justo sobre los ejes. El óptimo tiende a caer en una esquina — y una esquina significa un coeficiente exactamente cero.


## 5. Concepto mínimo

```text
Cresta (L2):  minimizar  RSS + α·Σ βⱼ²      → encoge, no anula
Lasso  (L1):  minimizar  RSS + α·Σ |βⱼ|     → ANULA

Umbral suave (la operación que lo produce):
    β ← signo(z) · máx(0, |z| − α·lr)
        si |z| ≤ α·lr  →  β = 0  exactamente
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('lasso', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos coeficientes dejará el lasso exactamente en cero?
2. ¿Y la regresión de cresta?
3. ¿Acertará el lasso qué variables son irrelevantes?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('lasso', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('lasso', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

De ocho variables, solo tres tienen efecto real. El lasso deja **4 coeficientes exactamente en cero** y la cresta, **0**: encoge todos pero no anula ninguno. Y de las cinco variables verdaderamente nulas, el lasso identifica cuatro.


## 10. Comentario pedagógico

Selecciona y estima en la misma operación, que es lo que lo hizo tan influyente. La misma idea reaparece treinta años después en [LoRA](../../papers/foundational/P48_lora/README.md): restringir el espacio de soluciones para obtener algo más simple y manejable, en vez de ajustar sin restricción y podar después.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer un coeficiente en cero como «esta variable no influye».


In [ ]:
print('Cero significa: dadas LAS DEMAS variables y ESTA penalizacion, no aporta.')
print('Con dos variables muy correlacionadas, el lasso se queda con una casi al azar.')
print('Es una afirmacion condicional sobre el modelo, no causal sobre el mundo.')

## 12. Corrección

La lectura correcta, con el camino de regularización delante:


In [ ]:
r = run_paper_lab('lasso', seed=7)['result']
for paso in r['camino_de_regularizacion']:
    print(f"alpha={paso['alpha']:<6} variables vivas={paso['no_nulos']}")
print('El conjunto seleccionado DEPENDE de alpha. Elegir alpha es parte del modelo.')

## 13. Desafío guiado

Compara los coeficientes de la variable 4 —copia ruidosa de la primera— en las tres soluciones y explica qué hace cada penalización con ella.


In [ ]:
r = run_paper_lab('lasso', seed=3)['result']
show(r)

## 14. Desafío autónomo

Ajusta un lasso sobre datos reales con validación cruzada para elegir alpha, y compáralo con una red elástica. Documenta qué variables sobreviven en cada caso y si el conjunto es estable al cambiar la partición.


## 15. Evidencia de aprendizaje

Guarda los tres vectores de coeficientes y el camino de regularización, con tu criterio para elegir alpha.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P77_lasso/README.md) · evaluación formal: [`assessments/papers/P77_lasso.md`](../../assessments/papers/P77_lasso.md)


## 16. Cierre

Un modelo con menos variables es más fácil de defender. La vía opuesta —muchos modelos malos combinados— resulta funcionar igual de bien.


## 17. Conexión con el siguiente hito

- P48
- P81

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
